# Reproduce — LeNet-5 / MNIST
Trains the two compressed models (DMC-Ultra and DMC-Board) using the
configs found by `search.ipynb`, evaluates them, generates C headers,
and compiles to report binary size for the ATmega328P (Arduino Uno).


In [ ]:
import os, sys, json, torch, subprocess
from torch import nn

_HERE      = os.path.dirname(os.path.abspath('.'))
_PROJ_ROOT = os.path.abspath(os.path.join('.', '../../..'))
_SHARED    = os.path.abspath(os.path.join('.', '../shared'))
sys.path.insert(0, _PROJ_ROOT)
sys.path.insert(0, _SHARED)

from development.experiments.lenet5  import get_model
from development.experiments.mnist   import get_data_loaders, get_metric
from train_utils import train_compressed, evaluate_model, save_results, print_results_table


In [ ]:
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED        = 25
INPUT_SHAPE = (1, 28, 28)
MODELS_DIR  = 'models'
DEPLOY_DIR  = 'deployment'
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DEPLOY_DIR, exist_ok=True)

torch.manual_seed(SEED)
print(f'Device: {DEVICE}')


## 1 — Data and baseline model

In [ ]:
print('Loading dataset …')
train_loader, test_loader = get_data_loaders()
metric_fn        = get_metric()
calibration_data = next(iter(train_loader))[0].to(DEVICE)

print('Loading model …')
baseline_model = get_model().to(DEVICE)


## 2 — Baseline model (FP32)

In [ ]:
# LeNet-5 has no pretrained weights — train from scratch.
baseline_ckpt = os.path.join(MODELS_DIR, 'baseline.pth')
if os.path.exists(baseline_ckpt):
    print(f'Loading baseline from {baseline_ckpt}')
    baseline_model.load_state_dict(
        torch.load(baseline_ckpt, weights_only=True)['model']
    )
else:
    print('Training baseline for 30 epochs …')
    from shared.train_utils import train_baseline
    train_baseline(baseline_model, train_loader, test_loader, metric_fn,
                   epochs=30, device=DEVICE)
    os.makedirs(MODELS_DIR, exist_ok=True)
    torch.save({'model': baseline_model.state_dict()}, baseline_ckpt)
    print(f'Baseline saved → {baseline_ckpt}')


In [ ]:
print('Evaluating baseline …')
baseline_results = evaluate_model(
    baseline_model, test_loader, metric_fn, INPUT_SHAPE, DEVICE
)
print('Baseline:', baseline_results)


## 3 — Load compression configs from search.ipynb

In [ ]:
ultra_config = torch.load(os.path.join(MODELS_DIR, 'dmc_ultra_config.pth'),
                          weights_only=False)
board_config = torch.load(os.path.join(MODELS_DIR, 'dmc_board_config.pth'),
                          weights_only=False)
print('Configs loaded.')


## 4 — Train DMC-Ultra

In [ ]:
print('Training DMC-Ultra …')
dmc_ultra = train_compressed(
    baseline_model, ultra_config, INPUT_SHAPE,
    train_loader, test_loader, metric_fn, calibration_data,
   epochs=40, device=DEVICE, two_step=True,
)
torch.save(dmc_ultra.state_dict(), os.path.join(MODELS_DIR, 'dmc_ultra.pth'))
ultra_results = evaluate_model(dmc_ultra, test_loader, metric_fn, INPUT_SHAPE, DEVICE)
print('DMC-Ultra:', ultra_results)


## 5 — Train DMC-Board

In [ ]:
print('Training DMC-Board …')
dmc_board = train_compressed(
    baseline_model, board_config, INPUT_SHAPE,
    train_loader, test_loader, metric_fn, calibration_data,
   epochs=40, device=DEVICE, two_step=True,
)
torch.save(dmc_board.state_dict(), os.path.join(MODELS_DIR, 'dmc_board.pth'))
board_results = evaluate_model(dmc_board, test_loader, metric_fn, INPUT_SHAPE, DEVICE)
print('DMC-Board:', board_results)


## 6 — Results summary

In [ ]:
all_results = {
    'Baseline (FP32)': baseline_results,
    'DMC-Ultra':       ultra_results,
    'DMC-Board':       board_results,
}
print_results_table(all_results)
save_results(all_results, MODELS_DIR)


In [ ]:
base_size = baseline_results['size_bytes']
base_ws   = baseline_results['workspace_bytes']
for name, r in all_results.items():
    cr  = base_size / r['size_bytes']   if r['size_bytes']   else float('inf')
    wsr = base_ws   / r['workspace_bytes'] if r['workspace_bytes'] else float('inf')
    print(f"{name:<20} size_CR={cr:.1f}x  workspace_CR={wsr:.1f}x")


## 7 — Generate C headers for deployment

In [ ]:
import random, string
test_input = torch.rand(INPUT_SHAPE, device=DEVICE)

for model_name, model in [('dmc_ultra', dmc_ultra), ('dmc_board', dmc_board)]:
    out_dir = os.path.join(DEPLOY_DIR, model_name)
    os.makedirs(out_dir, exist_ok=True)
    fused = model.fuse(device=DEVICE)
    fused.convert_to_c(
        INPUT_SHAPE, 'lenet5_model',
        out_dir, out_dir,
        for_arduino=True,
        test_input=test_input,
    )
    print(f'C headers written to {out_dir}/')


## 8 — Compile with avr-gcc and report binary size

In [ ]:
# Requires avr-gcc to be installed: sudo apt install gcc-avr binutils-avr
import glob

def avr_compile(src_dir, mcu='atmega2560'):
    c_files = glob.glob(os.path.join(src_dir, '*.cpp')) + \
              glob.glob(os.path.join(src_dir, '*.c'))
    if not c_files:
        print(f'  No C/C++ files found in {src_dir}'); return
    out_elf = os.path.join(src_dir, 'model.elf')
    cmd = ['avr-g++', f'-mmcu={mcu}', '-Os', '-o', out_elf] + c_files
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print('avr-g++ error:', result.stderr[:500]); return
    size_out = subprocess.run(['avr-size', out_elf],
                              capture_output=True, text=True).stdout
    print(f'\n{src_dir}:')
    print(size_out)

for model_name in ['dmc_ultra', 'dmc_board']:
    avr_compile(os.path.join(DEPLOY_DIR, model_name))
